# Erosion rate uncertainty calculations for CT samples

## Overview

**Purpose**: Compute subwatershed erosion rates by mass balance through the drainage network, propagating 13% external uncertainty (Bierman et al.) via Monte Carlo sampling.

**Inputs**:
- `MataquitoSampleData.xlsx` via `mataquito.sample_data` — basin-averaged erosion rates and drainage areas
- Flow network topology via `mataquito.flow_network` — defines parent-child relationships between sample sites

**Method**: For each non-headwater site, the subwatershed erosion rate is calculated as:

E_sub = (E_downstream × A_downstream − Σ E_upstream × A_upstream) / A_sub

where A_sub is the area between the downstream gauge and its upstream parents. The notebook walks through each node in the drainage hierarchy (CT-5, CT-6, CT-10, CT-11, CT-8, CT-2, CT-9) and shows the flux breakdown.

**Key result**: Some subwatersheds yield negative rates, indicating net sediment storage or deposition rather than erosion.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')

from mataquito.sample_data import load_samples, get_erosion_rates, get_areas
from mataquito.flow_network import all_subwatershed_erosion_rates, HEADWATERS

Subcatchment erosion rates for CT-6,5,4,10,11,8,2,9

In [2]:
# Load sample data from the canonical Excel file
df = load_samples()
E = get_erosion_rates(df)
A = get_areas(df)

print("Loaded from MataquitoSampleData.xlsx:")
for sid in sorted(E.keys(), key=lambda x: int(x.split('-')[1])):
    print(f"  {sid}: E = {E[sid]:.1f} m/Myr, A = {A[sid]:.2f} km²")

Loaded from MataquitoSampleData.xlsx:
  CT-1: E = 22.9 m/Myr, A = 1385.18 km²
  CT-2: E = 94.8 m/Myr, A = 5760.13 km²
  CT-3: E = 29.5 m/Myr, A = 189.21 km²
  CT-4: E = 29.6 m/Myr, A = 4706.72 km²
  CT-5: E = 387.0 m/Myr, A = 1495.90 km²
  CT-6: E = 29.8 m/Myr, A = 2573.24 km²
  CT-7: E = 532.0 m/Myr, A = 1207.81 km²
  CT-8: E = 263.0 m/Myr, A = 4950.40 km²
  CT-9: E = 105.0 m/Myr, A = 6189.72 km²
  CT-10: E = 246.0 m/Myr, A = 4864.94 km²
  CT-11: E = 286.0 m/Myr, A = 4913.33 km²


In [3]:
# Compute all subwatershed erosion rates using mass balance
# Flow network: CT-7→CT-5, CT-1→CT-6, CT-5+CT-6→CT-10→CT-11→CT-8→CT-2, CT-2+CT-3→CT-9
# Note: CT-4 is skipped (problematic data), lumped into CT-10 calculation

E_sub = all_subwatershed_erosion_rates(E, A)

print("=" * 70)
print("SUBWATERSHED EROSION RATES (m/Myr)")
print("=" * 70)
print("\nHeadwater subcatchments (original values):")
for hw in HEADWATERS:
    print(f"  {hw}: {E_sub[hw]:.2f}")
print("\nCalculated subwatersheds:")
for site in ['CT-5', 'CT-6', 'CT-10', 'CT-11', 'CT-8', 'CT-2', 'CT-9']:
    print(f"  {site}: {E_sub[site]:.2f}")
print(f"\n  CT-4: {E['CT-4']:.2f} (original - not used in hierarchy)")
print("\nNote: Negative values indicate net sediment storage/deposition")
print("=" * 70)

SUBWATERSHED EROSION RATES (m/Myr)

Headwater subcatchments (original values):
  CT-1: 22.90
  CT-3: 29.50
  CT-7: 532.00

Calculated subwatersheds:
  CT-5: -220.92
  CT-6: 37.84
  CT-10: 680.05
  CT-11: 4307.28
  CT-8: -2786.21
  CT-2: -933.51
  CT-9: 408.84

  CT-4: 29.60 (original - not used in hierarchy)

Note: Negative values indicate net sediment storage/deposition


In [4]:
# Detailed subwatershed calculation with flux breakdown
from mataquito.flow_network import FLOW_NETWORK, TRAVERSAL_ORDER, subwatershed_area

print("=" * 70)
print("DETAILED SUBWATERSHED EROSION RATE CALCULATIONS")
print("=" * 70)

for site in TRAVERSAL_ORDER:
    parents = FLOW_NETWORK[site]
    A_sub = subwatershed_area(site, A)
    upstream_flux = sum(E[p] * A[p] for p in parents)
    site_flux = E[site] * A[site]
    print(f"\n{'-'*70}")
    print(f"{site} (receives from {' + '.join(parents)}):")
    for p in parents:
        print(f"  Flux from {p}: {E[p] * A[p]:.2f}")
    if len(parents) > 1:
        print(f"  Total upstream flux: {upstream_flux:.2f}")
    print(f"  Flux at {site}: {site_flux:.2f}")
    print(f"  Subwatershed area: {A_sub:.2f} km²")
    print(f"  E_sub = {E_sub[site]:.2f} M/myr")

DETAILED SUBWATERSHED EROSION RATE CALCULATIONS

----------------------------------------------------------------------
CT-5 (receives from CT-7):
  Flux from CT-7: 642554.92
  Flux at CT-5: 578912.14
  Subwatershed area: 288.09 km²
  E_sub = -220.92 M/myr

----------------------------------------------------------------------
CT-6 (receives from CT-1):
  Flux from CT-1: 31720.74
  Flux at CT-6: 76682.70
  Subwatershed area: 1188.06 km²
  E_sub = 37.84 M/myr

----------------------------------------------------------------------
CT-10 (receives from CT-5 + CT-6):
  Flux from CT-5: 578912.14
  Flux from CT-6: 76682.70
  Total upstream flux: 655594.84
  Flux at CT-10: 1196775.73
  Subwatershed area: 795.80 km²
  E_sub = 680.05 M/myr

----------------------------------------------------------------------
CT-11 (receives from CT-10):
  Flux from CT-10: 1196775.73
  Flux at CT-11: 1405213.52
  Subwatershed area: 48.39 km²
  E_sub = 4307.28 M/myr

--------------------------------------------

### CT-5 Subwatershed Area

In [5]:
# CT-5 subwatershed (receives from CT-7)
Ea = E['CT-5']  # Downstream erosion rate
Eb = E['CT-7']  # Upstream erosion rate
Aa = A['CT-5']  # Downstream area
Ab = A['CT-7']  # Upstream area
Asub = Aa - Ab  # Subwatershed area

# Calculate using Ēsub = (ĒdownstreamAdownstream - ĒupstreamAupstream) / Asub
Ec_CT5 = (Ea * Aa - Eb * Ab) / Asub

print(f"CT-5 subwatershed erosion rate: {Ec_CT5:.2f}")
print(f"Downstream (CT-5): {Ea:.2f}")
print(f"Upstream (CT-7): {Eb:.2f}")
print(f"Downstream area: {Aa:.2f}")
print(f"Upstream area: {Ab:.2f}")
print(f"Subwatershed area: {Asub:.2f}")

CT-5 subwatershed erosion rate: -220.92
Downstream (CT-5): 387.00
Upstream (CT-7): 532.00
Downstream area: 1495.90
Upstream area: 1207.81
Subwatershed area: 288.09


### CT-6 Subwatershed area

In [6]:
# CT-6 subwatershed (receives from CT-1)
Ea = E['CT-6']  # Downstream erosion rate
Eb = E['CT-1']  # Upstream erosion rate
Aa = A['CT-6']  # Downstream area
Ab = A['CT-1']  # Upstream area
Asub = Aa - Ab  # Subwatershed area

# Calculate using Ēsub = (ĒdownstreamAdownstream - ĒupstreamAupstream) / Asub
Ec_CT6 = (Ea * Aa - Eb * Ab) / Asub

print(f"CT-6 subwatershed erosion rate: {Ec_CT6:.2f}")
print(f"Downstream (CT-6): {Ea:.2f}")
print(f"Upstream (CT-1): {Eb:.2f}")
print(f"Downstream area: {Aa:.2f}")
print(f"Upstream area: {Ab:.2f}")
print(f"Subwatershed area: {Asub:.2f}")

CT-6 subwatershed erosion rate: 37.84
Downstream (CT-6): 29.80
Upstream (CT-1): 22.90
Downstream area: 2573.24
Upstream area: 1385.18
Subwatershed area: 1188.06
